In [0]:
WITH date_bounds AS (
    SELECT
        MIN(valid_from) AS earliest,
        MAX(valid_to)   AS latest
    FROM edsm.silver.v_powerplay
    WHERE valid_to != '9999-12-31'
),
changing_keys AS (
    SELECT id, power
    FROM edsm.silver.v_powerplay
    WHERE valid_to != '9999-12-31'
    GROUP BY id, power
    HAVING COUNT(DISTINCT struct(powerState, state)) > 1
)
SELECT
    v.valid_from,
    v.valid_to,
    v.name,
    v.power,
    v.allegiance,
    v.government,
    v.powerState,
    v.state
FROM edsm.silver.v_powerplay v
WHERE (v.id, v.power) IN (SELECT id, power FROM changing_keys)
AND v.valid_from BETWEEN (SELECT earliest FROM date_bounds) AND (SELECT latest FROM date_bounds)
ORDER BY v.id, v.power, v.valid_from;

In [0]:
%python

dbutils.widgets.text("system_name", "")

system_name = dbutils.widgets.get("system_name")

latest_valid = spark.sql("SELECT MAX(valid_to) AS latest FROM edsm.silver.v_powerplay WHERE valid_to != '9999-12-31'").collect()[0]["latest"]

query = f"""
SELECT
    valid_from,
    CASE
        WHEN valid_to = '9999-12-31' THEN TIMESTAMP('{latest_valid}')
        ELSE valid_to
    END AS valid_to,
    powerState,
    state,
    collect_list(power) AS powers
FROM edsm.silver.v_powerplay
WHERE name = '{system_name}'
GROUP BY valid_from, valid_to, powerState, state
ORDER BY valid_from
"""

df = spark.sql(query).toPandas()

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df['valid_from'] = pd.to_datetime(df['valid_from'])
df['valid_to'] = pd.to_datetime(df['valid_to'])

all_powers = sorted(set(p for powers in df['powers'] for p in powers))
powers_label = ", ".join(all_powers)

fig, ax = plt.subplots(figsize=(12, len(df) * .8))

for i, row in df.iterrows():
    ax.barh(
        y=i,
        width=(row['valid_to'] - row['valid_from']).days,
        left=row['valid_from'],
        height=1.0
    )

ax.set_yticks(range(len(df)))
ax.set_yticklabels([f"{row['powerState']} - {row['state']}" for _, row in df.iterrows()])
ax.invert_yaxis()
ax.set_xlabel("Date")
ax.set_title(f"{system_name}")
fig.text(0.0, 0.98, f"Powers: {powers_label}", ha='left', va='top', fontsize=10, color='black')

ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
fig.autofmt_xdate(rotation=0)

plt.tight_layout()
plt.show()

In [0]:
SELECT
    name,
    COUNT(*) AS version_count,
    COUNT(DISTINCT power) AS distinct_powers
FROM edsm.silver.powerplay
WHERE current_flag = 'No'
GROUP BY name
HAVING COUNT(*) > 1
ORDER BY version_count DESC
LIMIT 25;